# Salinas Spatial Benchmark — GAST

This notebook adapts the **official `fesihkeskin/GAST` implementation** to the project's existing Salinas spatial split. The model architecture is imported from the official repository rather than reimplemented here.

**Benchmark protocol**
- Existing spatial split: `outputs/salinas/salinas_spatial_split_seed42.npz`
- Block size: 32 × 32; the notebook checks that blocks do not cross splits
- Centered patch size: 15 × 15
- Salinas corrected spectral bands and the project’s 16-class label mapping
- Training: AdamW, learning rate `1e-3`, weight decay `1e-4`, 50 epochs, cosine scheduler; best and latest checkpoints saved atomically after each completed epoch
- Best checkpoint selected by validation Macro-F1; test split used only for final evaluation
- Training loader batch: 128, gradient accumulation in microbatches of 16 to reduce GPU-memory pressure
- Efficiency benchmark: target batch 128, 20 warm-ups, 100 timed iterations; if GPU memory runs out, retries at a smaller batch and records the actual batch size

Validation and test evaluation use a smaller batch size of 16 to reduce memory pressure. The notebook writes the best checkpoint on every validation improvement and a resumable latest checkpoint after every epoch. If the kernel crashes, rerun cells in order; the training cell resumes from the latest compatible checkpoint. The notebook does **not** generate a new split. If the fixed split file is absent or fails the spatial-block check, stop and resolve that issue before training so model results remain comparable.

Official source: https://github.com/fesihkeskin/GAST


In [1]:
from pathlib import Path
import sys
import os
import json
import hashlib
import platform
import subprocess
import random
import shutil
import copy
import time
import importlib.util
import gc

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# Locate the project root using the actual project package layout.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / "src" / "dfm" / "data" / "salinas.py").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. Expected "
        "'src/dfm/data/salinas.py'. Open the notebook from the project "
        "root/notebooks directory or set PROJECT_ROOT manually."
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dfm.data.salinas import SALINAS_CLASS_NAMES, SalinasPatchDataset, load_salinas

try:
    import torch_geometric
except ImportError as exc:
    raise ImportError(
        "GAST requires PyTorch Geometric (`torch_geometric`). Install a build "
        "compatible with your installed PyTorch/CUDA version, then restart the kernel. "
        "See https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html"
    ) from exc

SEED = 42
PATCH_SIZE = 15
BLOCK_SIZE = 32
BATCH_SIZE = 128
MICRO_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
EPOCHS = 50
RESUME_IF_AVAILABLE = True
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
DROPOUT = 0.2

# Official GAST architecture configuration.
SPEC_DIM = 64
SPAT_DIM = 64
GAT_HEADS = 8
GAT_DEPTH = 4
TRANSFORMER_HEADS = 8
TRANSFORMER_LAYERS = 1
FUSION_MODE = "gate"

OUTPUTS = PROJECT_ROOT / "outputs" / "salinas"
OUTPUTS.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUTS / "gast_spatial_best.pt"
LATEST_CHECKPOINT_PATH = OUTPUTS / "gast_spatial_latest.pt"
HISTORY_PATH = OUTPUTS / "gast_spatial_history.csv"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT)
print("Device:", device)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup
Device: cuda
Python: 3.11.5
PyTorch: 2.5.1+cu118
PyTorch Geometric: 2.8.0.post1
GPU: NVIDIA GeForce RTX 2050


In [2]:
# Reproducibility controls.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Seed:", SEED)


Seed: 42


In [3]:
# Load the project's Salinas corrected scene.
data_dir = PROJECT_ROOT / "data" / "raw" / "salinas"
scene = load_salinas(data_dir, download=True)

rows, cols = np.where(scene.labels > 0)
all_labels = scene.labels[rows, cols].astype(np.int64) - 1

print("Cube (H, W, C):", scene.cube.shape)
print("Ground-truth map:", scene.labels.shape)
print("Spectral bands:", scene.bands)
print("Labeled pixels:", len(all_labels))
print("Classes:", len(SALINAS_CLASS_NAMES))

assert scene.bands == 204, f"Expected Salinas corrected with 204 bands, got {scene.bands}"
assert len(SALINAS_CLASS_NAMES) == 16, "Unexpected class-name mapping."
assert np.array_equal(np.unique(all_labels), np.arange(len(SALINAS_CLASS_NAMES))), (
    "Expected all 16 zero-indexed classes in the labeled pixels."
)


Cube (H, W, C): (512, 217, 204)
Ground-truth map: (512, 217)
Spectral bands: 204
Labeled pixels: 54129
Classes: 16


In [4]:
# Load ONLY the fixed spatial split. Do not silently create a random split.
SPLIT_PATH = OUTPUTS / "salinas_spatial_split_seed42.npz"
if not SPLIT_PATH.exists():
    raise FileNotFoundError(
        f"Required fixed spatial split not found: {SPLIT_PATH}\n"
        "Run the project's shared spatial-split benchmark notebook first."
    )

split = np.load(SPLIT_PATH)
required_keys = {"train_indices", "val_indices", "test_indices"}
if not required_keys.issubset(set(split.files)):
    raise KeyError(f"Split file must contain {sorted(required_keys)}; found {split.files}")

train_indices = np.asarray(split["train_indices"], dtype=np.int64)
val_indices = np.asarray(split["val_indices"], dtype=np.int64)
test_indices = np.asarray(split["test_indices"], dtype=np.int64)

for name, idx in [
    ("train", train_indices),
    ("validation", val_indices),
    ("test", test_indices),
]:
    if idx.ndim != 1 or len(idx) == 0:
        raise ValueError(f"{name} indices must be a non-empty 1-D array.")
    if idx.min() < 0 or idx.max() >= len(all_labels):
        raise IndexError(f"{name} indices are outside the labeled-pixel index range.")

if (set(train_indices) & set(val_indices)) or (set(train_indices) & set(test_indices)) or (set(val_indices) & set(test_indices)):
    raise ValueError("The saved spatial split contains overlapping sample indices.")

# Verify spatial separation at the same 32x32 block size used by the shared benchmark.
def block_set(indices):
    return set(zip((rows[indices] // BLOCK_SIZE).tolist(),
                   (cols[indices] // BLOCK_SIZE).tolist()))

train_blocks = block_set(train_indices)
val_blocks = block_set(val_indices)
test_blocks = block_set(test_indices)

if train_blocks & val_blocks or train_blocks & test_blocks or val_blocks & test_blocks:
    raise ValueError(
        "The loaded split has blocks shared between train/validation/test. "
        "Do not train until the shared spatial split is corrected."
    )

def sha256_array(array):
    return hashlib.sha256(np.ascontiguousarray(array).tobytes()).hexdigest()

split_hashes = {
    "train": sha256_array(train_indices),
    "validation": sha256_array(val_indices),
    "test": sha256_array(test_indices),
}

print("Split file:", SPLIT_PATH)
print("Split fingerprint:", json.dumps(split_hashes, indent=2))
print(f"Train: {len(train_indices):,} samples | {len(train_blocks)} blocks")
print(f"Validation: {len(val_indices):,} samples | {len(val_blocks)} blocks")
print(f"Test: {len(test_indices):,} samples | {len(test_blocks)} blocks")


Split file: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\salinas_spatial_split_seed42.npz
Split fingerprint: {
  "train": "ec3617b38356b1ad4e2b8445b345f87be519a53ca34626484cb93d7b48a171d6",
  "validation": "f569fc2e69cbf6bc770e170b9735459d7332d024fe06c433cde990c0f5287d8d",
  "test": "6fb69fcd80676361c70bb751f6f73361dfe8d083b655de33fa7d69424358617e"
}
Train: 32,337 samples | 49 blocks
Validation: 10,952 samples | 16 blocks
Test: 10,840 samples | 16 blocks


In [5]:
# Construct datasets using the exact shared split indices.
train_dataset = SalinasPatchDataset(
    scene=scene, indices=train_indices, patch_size=PATCH_SIZE, normalize=True
)
val_dataset = SalinasPatchDataset(
    scene=scene, indices=val_indices, patch_size=PATCH_SIZE, normalize=True
)
test_dataset = SalinasPatchDataset(
    scene=scene, indices=test_indices, patch_size=PATCH_SIZE, normalize=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

x_sample, y_sample = next(iter(train_loader))
print("Sample batch:", tuple(x_sample.shape), tuple(y_sample.shape))
print("Patch tensor convention: [batch, bands, height, width]")
assert tuple(x_sample.shape[1:]) == (scene.bands, PATCH_SIZE, PATCH_SIZE)


Sample batch: (128, 204, 15, 15) (128,)
Patch tensor convention: [batch, bands, height, width]


In [6]:
# Fetch/import the official GAST architecture.
# Keep the official repository in its own dedicated folder so it cannot collide with project `src`.
GAST_REPO_URL = "https://github.com/fesihkeskin/GAST.git"
external_dir = PROJECT_ROOT / "external"
external_dir.mkdir(parents=True, exist_ok=True)
gast_repo_dir = external_dir / "GAST"
gast_model_file = gast_repo_dir / "src" / "models" / "model_architecture.py"

if not gast_model_file.exists():
    if gast_repo_dir.exists():
        print("Existing GAST checkout is incomplete; replacing only external/GAST.")
        shutil.rmtree(gast_repo_dir)
    subprocess.run(
        ["git", "clone", "--depth", "1", GAST_REPO_URL, str(gast_repo_dir)],
        check=True,
    )
    gast_model_file = gast_repo_dir / "src" / "models" / "model_architecture.py"

if not gast_model_file.exists():
    raise FileNotFoundError(
        "Official GAST model file not found after checkout: "
        f"{gast_model_file}"
    )

spec = importlib.util.spec_from_file_location("gast_official_architecture", gast_model_file)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not create an import specification for {gast_model_file}")
gast_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = gast_module
spec.loader.exec_module(gast_module)
GAST = gast_module.GAST

print("Official model source:", gast_model_file)
print("Repository URL:", GAST_REPO_URL)


Official model source: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\external\GAST\src\models\model_architecture.py
Repository URL: https://github.com/fesihkeskin/GAST.git


In [7]:
# Instantiate GAST with the official implementation's public constructor.
MODEL_NAME = "gast"

model = GAST(
    in_channels=scene.bands,
    n_classes=len(SALINAS_CLASS_NAMES),
    patch_size=PATCH_SIZE,
    spec_dim=SPEC_DIM,
    spat_dim=SPAT_DIM,
    n_heads=GAT_HEADS,
    n_layers=GAT_DEPTH,
    transformer_heads=TRANSFORMER_HEADS,
    transformer_layers=TRANSFORMER_LAYERS,
    dropout=DROPOUT,
    disable_spectral=False,
    disable_spatial=False,
    fusion_mode=FUSION_MODE,
).to(device)

model.eval()
with torch.no_grad():
    logits_check = model(x_sample[:2].to(device=device, dtype=torch.float32))

print("Output for 2 samples:", tuple(logits_check.shape))
assert tuple(logits_check.shape) == (2, len(SALINAS_CLASS_NAMES)), (
    f"Expected (2, {len(SALINAS_CLASS_NAMES)}) logits, got {tuple(logits_check.shape)}"
)

n_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_parameters:,} ({n_parameters / 1e6:.4f} M)")
print(model)


Output for 2 samples: (2, 16)
Trainable parameters: 352,464 (0.3525 M)
GAST(
  (spectral_mlp): Sequential(
    (0): Linear(in_features=204, out_features=64, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=64, bias=True)
  )
  (spectral_transformer): TransformerEncoder(
    (layers): ModuleList(
      (0): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )

## Training and crash recovery

The train loader uses batches of 128, processed in microbatches of 16. Validation and test loaders use batches of 16 to reduce peak memory. At every completed epoch, the notebook atomically saves `outputs/salinas/gast_spatial_latest.pt` (model, optimizer, scheduler, RNG states, history and best state), and saves `gast_spatial_best.pt` immediately whenever validation Macro-F1 improves. The history CSV is also updated after every epoch.

If the kernel dies, restart the kernel and run cells in order. The training cell resumes from the latest compatible checkpoint rather than starting from epoch 1. Do not delete the checkpoint files if you want to resume.


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
)

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_samples = 0

    for x, y in tqdm(loader, desc="train", leave=False):
        x = x.to(device=device, dtype=torch.float32, non_blocking=(device.type == "cuda"))
        y = y.to(device=device, dtype=torch.long, non_blocking=(device.type == "cuda"))
        optimizer.zero_grad(set_to_none=True)
        batch_n = y.size(0)
        batch_loss_value = 0.0

        for start in range(0, batch_n, MICRO_BATCH_SIZE):
            end = min(start + MICRO_BATCH_SIZE, batch_n)
            xb, yb = x[start:end], y[start:end]
            logits = model(xb)
            loss = criterion(logits, yb)
            (loss * ((end - start) / batch_n)).backward()
            batch_loss_value += float(loss.detach().item()) * (end - start)
            del logits, loss, xb, yb

        optimizer.step()
        total_loss += batch_loss_value
        total_samples += batch_n
        del x, y

    return total_loss / max(total_samples, 1)


@torch.no_grad()
def evaluate(model, loader, return_predictions=False):
    model.eval()
    total_loss = 0.0
    total = 0
    y_true, y_pred = [], []

    for x, y in tqdm(loader, desc="evaluate", leave=False):
        x = x.to(device=device, dtype=torch.float32, non_blocking=(device.type == "cuda"))
        y = y.to(device=device, dtype=torch.long, non_blocking=(device.type == "cuda"))
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += float(loss.item()) * y.size(0)
        total += y.size(0)
        y_true.extend(y.cpu().numpy().tolist())
        y_pred.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        del x, y, logits, loss

    metrics = {
        "loss": total_loss / max(total, 1),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true, y_pred, average="macro",
            labels=np.arange(len(SALINAS_CLASS_NAMES)), zero_division=0
        ),
    }
    if return_predictions:
        return metrics, np.asarray(y_true, dtype=np.int64), np.asarray(y_pred, dtype=np.int64)
    return metrics


In [9]:
# Crash-safe training: save the best model immediately on every validation improvement,
# and save a resumable latest checkpoint after EVERY completed epoch.
import tempfile

RUN_SIGNATURE = {
    "model": MODEL_NAME,
    "seed": SEED,
    "split_hashes": split_hashes,
    "patch_size": PATCH_SIZE,
    "batch_size": BATCH_SIZE,
    "micro_batch_size": MICRO_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "epochs": EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "spec_dim": SPEC_DIM,
    "spat_dim": SPAT_DIM,
    "gat_heads": GAT_HEADS,
    "gat_depth": GAT_DEPTH,
    "transformer_heads": TRANSFORMER_HEADS,
    "transformer_layers": TRANSFORMER_LAYERS,
    "dropout": DROPOUT,
    "fusion_mode": FUSION_MODE,
}


def atomic_torch_save(payload, destination):
    destination = Path(destination)
    temp_path = destination.with_suffix(destination.suffix + ".tmp")
    torch.save(payload, temp_path)
    os.replace(temp_path, destination)


def save_history(rows):
    pd.DataFrame(rows).to_csv(HISTORY_PATH, index=False)


def make_checkpoint_payload(epoch_completed):
    return {
        "format_version": 2,
        "epoch_completed": int(epoch_completed),
        "run_signature": RUN_SIGNATURE,
        "model_state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
        "best_state": best_state,
        "best_epoch": int(best_epoch),
        "best_val_macro_f1": float(best_val_macro_f1),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "history": history,
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }

best_val_macro_f1 = -1.0
best_epoch = 0
best_state = None
history = []
start_epoch = 1

if RESUME_IF_AVAILABLE and LATEST_CHECKPOINT_PATH.exists():
    print("Found latest checkpoint:", LATEST_CHECKPOINT_PATH)
    # weights_only=False is needed because optimizer/RNG/history states are included.
    # Only load this checkpoint if it is a file created by this notebook in your own project.
    saved = torch.load(LATEST_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    if saved.get("run_signature") != RUN_SIGNATURE:
        raise RuntimeError(
            "A latest checkpoint exists, but its run signature does not match this run. "
            "It was NOT loaded. Move/rename the old checkpoint if you intend a fresh run."
        )
    model.load_state_dict(saved["model_state_dict"])
    optimizer.load_state_dict(saved["optimizer_state_dict"])
    scheduler.load_state_dict(saved["scheduler_state_dict"])
    best_state = saved["best_state"]
    best_epoch = int(saved["best_epoch"])
    best_val_macro_f1 = float(saved["best_val_macro_f1"])
    history = list(saved["history"])
    start_epoch = int(saved["epoch_completed"]) + 1
    random.setstate(saved["python_rng_state"])
    np.random.set_state(saved["numpy_rng_state"])
    torch.set_rng_state(saved["torch_rng_state"])
    if torch.cuda.is_available() and saved.get("cuda_rng_state_all") is not None:
        torch.cuda.set_rng_state_all(saved["cuda_rng_state_all"])
    model.to(device)
    print(f"Resuming at epoch {start_epoch}/{EPOCHS}; best epoch so far: {best_epoch}")
else:
    print("No compatible latest checkpoint found; starting from epoch 1.")

for epoch in range(start_epoch, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader)
    val_metrics = evaluate(model, val_loader)
    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "lr": optimizer.param_groups[0]["lr"],
    }
    history.append(row)

    if val_metrics["macro_f1"] > best_val_macro_f1:
        best_val_macro_f1 = float(val_metrics["macro_f1"])
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        # Persist the best weights immediately, before moving on to another epoch.
        atomic_torch_save({
            "format_version": 2,
            "model_state_dict": best_state,
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_val_macro_f1,
            "run_signature": RUN_SIGNATURE,
            "split_artifact": SPLIT_PATH.name,
            "official_repository": GAST_REPO_URL,
        }, CHECKPOINT_PATH)
        print(f"  Saved BEST checkpoint: epoch {best_epoch}, val Macro-F1 {best_val_macro_f1:.5f}")

    # Persist history and the complete resumable state at every epoch boundary.
    save_history(history)
    atomic_torch_save(make_checkpoint_payload(epoch), LATEST_CHECKPOINT_PATH)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | Train loss {train_loss:.5f} | "
        f"Val OA {val_metrics['accuracy']:.5f} | Val Macro-F1 {val_metrics['macro_f1']:.5f}"
    )
    if device.type == "cuda":
        torch.cuda.empty_cache()
    gc.collect()

if best_state is None:
    # If training had already completed and only best checkpoint exists, load it.
    if CHECKPOINT_PATH.exists():
        best_payload = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
        if best_payload.get("run_signature") != RUN_SIGNATURE:
            raise RuntimeError("Best checkpoint signature mismatch; refusing to load it.")
        best_state = best_payload["model_state_dict"]
        best_epoch = int(best_payload["best_epoch"])
        best_val_macro_f1 = float(best_payload["best_val_macro_f1"])
    else:
        raise RuntimeError("No best checkpoint was produced. Check the error above.")

model.load_state_dict(best_state)
model.to(device)
print("Best epoch:", best_epoch)
print("Best validation Macro-F1:", best_val_macro_f1)
print("Best checkpoint:", CHECKPOINT_PATH)
print("Latest resumable checkpoint:", LATEST_CHECKPOINT_PATH)


No compatible latest checkpoint found; starting from epoch 1.


  Saved BEST checkpoint: epoch 1, val Macro-F1 0.81640
Epoch 01/50 | Train loss 0.49941 | Val OA 0.80835 | Val Macro-F1 0.81640


  Saved BEST checkpoint: epoch 2, val Macro-F1 0.87526
Epoch 02/50 | Train loss 0.10843 | Val OA 0.89573 | Val Macro-F1 0.87526


  Saved BEST checkpoint: epoch 3, val Macro-F1 0.89136
Epoch 03/50 | Train loss 0.07787 | Val OA 0.92695 | Val Macro-F1 0.89136


Epoch 04/50 | Train loss 0.04579 | Val OA 0.92248 | Val Macro-F1 0.88527


...

Epoch 46/50 | Train loss 0.00016 | Val OA 0.90714 | Val Macro-F1 0.87827


Epoch 47/50 | Train loss 0.00005 | Val OA 0.90604 | Val Macro-F1 0.87713


Epoch 48/50 | Train loss 0.00010 | Val OA 0.90632 | Val Macro-F1 0.87994


Epoch 49/50 | Train loss 0.00004 | Val OA 0.90623 | Val Macro-F1 0.87980


Epoch 50/50 | Train loss 0.00008 | Val OA 0.90641 | Val Macro-F1 0.87993
Best epoch: 6
Best validation Macro-F1: 0.9305620376245523
Best checkpoint: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\gast_spatial_best.pt
Latest resumable checkpoint: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\gast_spatial_latest.pt


In [10]:
# Final test evaluation: the test split has not been used for model selection.
test_metrics, y_test, y_pred = evaluate(
    model, test_loader, return_predictions=True
)
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(len(SALINAS_CLASS_NAMES)),
)
report = classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(SALINAS_CLASS_NAMES)),
    target_names=list(SALINAS_CLASS_NAMES),
    output_dict=True,
    zero_division=0,
)

print("=" * 64)
print("GAST — FINAL SPATIAL TEST RESULTS")
print("=" * 64)
print(f"Best epoch : {best_epoch}")
print(f"Accuracy   : {test_metrics['accuracy']:.6f}")
print(f"Macro-F1   : {test_metrics['macro_f1']:.6f}")
print(f"Test loss  : {test_metrics['loss']:.6f}")
print()
print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(SALINAS_CLASS_NAMES)),
    target_names=list(SALINAS_CLASS_NAMES),
    digits=4,
    zero_division=0,
))


GAST — FINAL SPATIAL TEST RESULTS
Best epoch : 6
Accuracy   : 0.957657
Macro-F1   : 0.928503
Test loss  : 0.211952

                           precision    recall  f1-score   support

    Brocoli_green_weeds_1     1.0000    0.9692    0.9844       909
    Brocoli_green_weeds_2     1.0000    1.0000    1.0000       498
                   Fallow     0.7624    0.9865    0.8601       296
        Fallow_rough_plow     0.9933    1.0000    0.9966       295
            Fallow_smooth     1.0000    0.9977    0.9988       863
                  Stubble     0.9890    1.0000    0.9945       632
                   Celery     0.8108    1.0000    0.8955       120
         Grapes_untrained     0.9760    0.9417    0.9586      2899
     Soil_vinyard_develop     0.9874    1.0000    0.9937      1177
Corn_senesced_green_weeds     0.9704    0.6618    0.7869       544
      Lettuce_romaine_4wk     0.4435    1.0000    0.6145        51
      Lettuce_romaine_5wk     1.0000    1.0000    1.0000       448
      Lettuc

In [11]:
# Save reproducibility artifacts and final metrics.
CHECKPOINT_PATH = OUTPUTS / "gast_spatial_best.pt"
HISTORY_PATH = OUTPUTS / "gast_spatial_history.csv"
PREDICTIONS_PATH = OUTPUTS / "gast_spatial_predictions.csv"
CONFUSION_PATH = OUTPUTS / "gast_spatial_confusion_matrix.csv"
RESULTS_PATH = OUTPUTS / "gast_spatial_results.csv"
REPORT_PATH = OUTPUTS / "gast_spatial_classification_report.csv"
CONFIG_PATH = OUTPUTS / "gast_spatial_run_config.json"

# The best checkpoint was saved atomically during training. Do not overwrite it here.
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Best checkpoint missing: {CHECKPOINT_PATH}")

pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
pd.DataFrame({
    "y_true": y_test,
    "y_pred": y_pred,
}).to_csv(PREDICTIONS_PATH, index=False)
pd.DataFrame(
    cm,
    index=SALINAS_CLASS_NAMES,
    columns=SALINAS_CLASS_NAMES,
).to_csv(CONFUSION_PATH)
pd.DataFrame(report).transpose().to_csv(REPORT_PATH)

result_row = {
    "model": MODEL_NAME,
    "split": "spatial",
    "seed": SEED,
    "best_epoch": best_epoch,
    "accuracy": test_metrics["accuracy"],
    "macro_f1": test_metrics["macro_f1"],
    "test_loss": test_metrics["loss"],
    "parameters": n_parameters,
    "parameters_m": n_parameters / 1e6,
}
pd.DataFrame([result_row]).to_csv(RESULTS_PATH, index=False)

run_config = {
    "model": MODEL_NAME,
    "official_repository": GAST_REPO_URL,
    "official_model_file": str(gast_model_file),
    "seed": SEED,
    "split_artifact": SPLIT_PATH.name,
    "split_hashes": split_hashes,
    "block_size": BLOCK_SIZE,
    "patch_size": PATCH_SIZE,
    "batch_size": BATCH_SIZE,
    "micro_batch_size": MICRO_BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "AdamW",
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "CosineAnnealingLR(T_max=EPOCHS)",
    "best_checkpoint_selected_by": "validation macro-F1",
    "test_metrics": test_metrics,
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "device": str(device),
}
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

print("Saved:")
for path in [
    CHECKPOINT_PATH, HISTORY_PATH, PREDICTIONS_PATH,
    CONFUSION_PATH, RESULTS_PATH, REPORT_PATH, CONFIG_PATH
]:
    print(" -", path, "| exists:", path.exists())


Saved:
 - c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\gast_spatial_best.pt | exists: True
 - c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\gast_spatial_history.csv | exists: True
 - c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\gast_spatial_predictions.csv | exists: True
 - c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\gast_spatial_confusion_matrix.csv | exists: True
 - c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\gast_spatial_results.csv | exists: True
 - c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\gast_spati

## Efficiency benchmark

The benchmark runs on the trained model in evaluation mode. FLOPs profiling is isolated to a deep copy because profiling hooks can alter module state; PyTorch Geometric operations may not be fully counted by THOP, so a failed or incomplete FLOPs estimate is saved as unavailable rather than treated as a reliable zero.


In [12]:
# Count parameters and checkpoint size.
parameter_count = sum(p.numel() for p in model.parameters())
trainable_parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
checkpoint_size_mb = CHECKPOINT_PATH.stat().st_size / (1024 ** 2)

# FLOPs: profile a deep copy on a single example to avoid contaminating the live model.
flops = np.nan
gflops = np.nan
try:
    from thop import profile
    profile_model = copy.deepcopy(model).to(device).eval()
    profile_input = torch.zeros(
        1, scene.bands, PATCH_SIZE, PATCH_SIZE,
        device=device, dtype=torch.float32
    )
    with torch.no_grad():
        flops_value, _ = profile(profile_model, inputs=(profile_input,), verbose=False)
    flops = float(flops_value)
    gflops = flops / 1e9
    del profile_model, profile_input
    print("THOP FLOPs (one sample):", flops)
except Exception as exc:
    print("FLOPs unavailable from THOP for this GAST/PyG model:", repr(exc))
    flops = np.nan
    gflops = np.nan
finally:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"Parameters: {parameter_count:,}")
print(f"Trainable parameters: {trainable_parameter_count:,}")
print(f"Checkpoint size: {checkpoint_size_mb:.3f} MB")


THOP FLOPs (one sample): 9514496.0
Parameters: 352,464
Trainable parameters: 352,464
Checkpoint size: 1.360 MB


In [13]:
# Benchmark batch inference. Try the common batch size first, then safely fall back on OOM.
model.eval()
candidate_batch_sizes = [128, 64, 32, 16, 8, 1]
benchmark_batch = None
benchmark_batch_size = None

for candidate in candidate_batch_sizes:
    try:
        items = [test_dataset[i] for i in range(min(candidate, len(test_dataset)))]
        xb = torch.stack([torch.as_tensor(item[0], dtype=torch.float32) for item in items]).to(device)
        with torch.no_grad():
            _ = model(xb)
        benchmark_batch = xb
        benchmark_batch_size = len(items)
        print("Benchmark batch selected:", benchmark_batch_size)
        break
    except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:
        if isinstance(exc, RuntimeError) and "out of memory" not in str(exc).lower() and not isinstance(exc, torch.cuda.OutOfMemoryError):
            raise
        print(f"CUDA OOM at batch {candidate}; retrying smaller batch.")
        if "xb" in locals():
            del xb
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

if benchmark_batch is None:
    raise RuntimeError("Could not run a forward pass even with batch size 1.")

# Warm-up and timed iterations.
WARMUP_RUNS = 20
TIMED_RUNS = 100
latencies_ms = []

with torch.no_grad():
    for _ in range(WARMUP_RUNS):
        _ = model(benchmark_batch)

    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats(device)

    for _ in range(TIMED_RUNS):
        if device.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        _ = model(benchmark_batch)
        if device.type == "cuda":
            torch.cuda.synchronize()
        latencies_ms.append((time.perf_counter() - start) * 1000.0)

latencies_ms = np.asarray(latencies_ms, dtype=np.float64)
latency_mean = float(np.mean(latencies_ms))
latency_median = float(np.median(latencies_ms))
latency_p95 = float(np.percentile(latencies_ms, 95))
latency_std = float(np.std(latencies_ms))
latency_per_sample = latency_mean / benchmark_batch_size
throughput = (benchmark_batch_size * 1000.0) / latency_mean
peak_gpu_memory_mb = (
    torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    if device.type == "cuda" else np.nan
)

efficiency_row = {
    "model": MODEL_NAME,
    "accuracy": test_metrics["accuracy"],
    "macro_f1": test_metrics["macro_f1"],
    "parameters": parameter_count,
    "parameters_m": parameter_count / 1e6,
    "model_size_mb": checkpoint_size_mb,
    "flops": flops,
    "gflops": gflops,
    "peak_gpu_memory_mb": peak_gpu_memory_mb,
    "batch_size": benchmark_batch_size,
    "latency_batch_ms": latency_mean,
    "latency_batch_median_ms": latency_median,
    "latency_batch_p95_ms": latency_p95,
    "latency_batch_std_ms": latency_std,
    "latency_per_sample_ms": latency_per_sample,
    "throughput_samples_sec": throughput,
    "warmup_runs": WARMUP_RUNS,
    "timed_runs": TIMED_RUNS,
}
EFFICIENCY_PATH = OUTPUTS / "gast_spatial_efficiency.csv"
pd.DataFrame([efficiency_row]).to_csv(EFFICIENCY_PATH, index=False)

print("Efficiency benchmark:")
print(pd.DataFrame([efficiency_row]).T.to_string(header=False))
print("Saved:", EFFICIENCY_PATH)


Benchmark batch selected: 128
Efficiency benchmark:
model                           gast
accuracy                    0.957657
macro_f1                    0.928503
parameters                    352464
parameters_m                0.352464
model_size_mb                1.36034
flops                      9514496.0
gflops                      0.009514
peak_gpu_memory_mb       2035.927734
batch_size                       128
latency_batch_ms          797.562598
latency_batch_median_ms     765.1303
latency_batch_p95_ms      1105.27767
latency_batch_std_ms      115.137926
latency_per_sample_ms       6.230958
throughput_samples_sec     160.48897
warmup_runs                       20
timed_runs                       100
Saved: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\gast_spatial_efficiency.csv


## Completion checklist

- [ ] The notebook loaded the existing `salinas_spatial_split_seed42.npz`; it did not create a new split.
- [ ] The spatial-block overlap check passed.
- [ ] The official GAST forward-pass check returned `(batch, 16)` logits.
- [ ] The best checkpoint was selected using validation Macro-F1.
- [ ] Final test metrics were computed once using the held-out spatial test set.
- [ ] `gast_spatial_results.csv` and `gast_spatial_efficiency.csv` were saved.

**Interpretation caution:** GAST’s graph-attention FLOPs may not be fully represented by generic profiling tools. Keep FLOPs marked unavailable if the profiler cannot support the relevant PyG operations. Also record the actual benchmark batch size if the GPU-memory fallback is triggered.
